In [1]:
import numpy as np
import pandas as pd

import string, faiss

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, set_seed
from sentence_transformers import SentenceTransformer

from peft import LoraConfig, TaskType, get_peft_model
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset

Set seed for reproducibility

In [2]:
set_seed(42)

Load exoplanet dataset

In [3]:
df=pd.read_csv('data/exoplanets.csv', index_col=0)

In [4]:
df.head(3)

,pl_name,hostname,discoverymethod,disc_year,disc_facility,pl_orbper,pl_rade,pl_bmasse,pl_eqt,pl_orbsmax,pl_orbeccen,st_teff,st_rad,st_mass,sy_dist,sy_vmag,ra,dec,rowupdate
0,TOI-4495 b,TOI-4495,Transit,2026.0,Transiting Exoplanet Survey Satellite (TESS),2.566990,2.483000,7.7,1735.0,0.03957,0.078,6210.00,1.309000,1.247,132.214,9.531,286.362495,37.025990,2026-02-12
1,TOI-4511 b,TOI-4511,Transit,2026.0,Transiting Exoplanet Survey Satellite (TESS),10.980910,2.753049,NaN,NaN,NaN,NaN,5418.77,1.001450,NaN,121.832,10.537,49.305279,15.501727,2026-04-23
2,TOI-4513 b,TOI-4513,Transit,2026.0,Transiting Exoplanet Survey Satellite (TESS),9.760423,2.345876,NaN,NaN,NaN,NaN,5623.93,0.796628,NaN,102.609,10.696,18.090518,13.755080,2026-04-23


In [5]:
# replace spaces with "-" in names
df['pl_name'] = df['pl_name'].apply(lambda x: x.replace(" ", "-"))
df['hostname'] = df['hostname'].apply(lambda x: x.replace(" ", "-"))
# convert discovery year to int
df['disc_year']=df['disc_year'].astype('Int32')

Create verbose description and schematic summary for RAG

In [6]:
# create verbose description of each entry
df['description'] = df.apply(lambda x: f"The planet {x.pl_name} orbits around the star {x.hostname}. It was discovered in {x.disc_year}. It has a mass of {x.pl_bmasse} Mass Earth Units. It has an orbital period of {x.pl_orbper} days. It has a radius of {x.pl_rade} Radius Earth Units. It has a temperature {x.pl_eqt} K", axis=1)
df['description'] = df['description'].apply(lambda x: x.replace(" nan ", " unknown "))

In [7]:
# create schematic summary of each entry
df['summary'] = df.apply(lambda x: f"name {x.pl_name} | host_name {x.hostname} | discovery_year {x.disc_year} | mass {x.pl_bmasse} | orbital_period {x.pl_orbper} | radius {x.pl_rade} | temperature {x.pl_eqt}", axis=1)
df['summary'] = df['summary'].apply(lambda x: x.replace(" nan ", " unknown "))

In [8]:
df.head(3)['description'].tolist()

['The planet TOI-4495-b orbits around the star TOI-4495. It was discovered in 2026. It has a mass of 7.7 Mass Earth Units. It has an orbital period of 2.56699 days. It has a radius of 2.483 Radius Earth Units. It has a temperature 1735.0 K',
 'The planet TOI-4511-b orbits around the star TOI-4511. It was discovered in 2026. It has a mass of unknown Mass Earth Units. It has an orbital period of 10.98090968596 days. It has a radius of 2.75304932 Radius Earth Units. It has a temperature unknown K',
 'The planet TOI-4513-b orbits around the star TOI-4513. It was discovered in 2026. It has a mass of unknown Mass Earth Units. It has an orbital period of 9.76042253684 days. It has a radius of 2.34587608 Radius Earth Units. It has a temperature unknown K']

In [9]:
df.head(3)['summary'].tolist()

['name TOI-4495-b | host_name TOI-4495 | discovery_year 2026 | mass 7.7 | orbital_period 2.56699 | radius 2.483 | temperature 1735.0',
 'name TOI-4511-b | host_name TOI-4511 | discovery_year 2026 | mass unknown | orbital_period 10.98090968596 | radius 2.75304932 | temperature nan',
 'name TOI-4513-b | host_name TOI-4513 | discovery_year 2026 | mass unknown | orbital_period 9.76042253684 | radius 2.34587608 | temperature nan']

RAG strategy 1: create vector embeddings for verbose description and define function to retrieve the 3 entries closest to the query by L2 norm

In [10]:
'''
# create and index vector embeddings
st_model = SentenceTransformer('all-MiniLM-L6-v2')
corpus = df['description'].tolist()
embeddings = st_model.encode(corpus, convert_to_numpy=True, show_progress_bar=True)
index = faiss.IndexFlatL2(embeddings.shape[1])
faiss.normalize_L2(embeddings)
index.add(embeddings)
faiss.write_index(index, 'exoplanet.index')
np.save('exoplanet.npy', embeddings)

#retrieve relevant information
def retrieve(query, k=3):
    embedding = st_model.encode([query], convert_to_numpy=True, show_progress_bar=False)
    faiss.normalize_L2(embedding)
    dst, idx = index.search(embedding, k)

    return df.iloc[idx[0]]['description'].tolist()
'''

"\n# create and index vector embeddings\nst_model = SentenceTransformer('all-MiniLM-L6-v2')\ncorpus = df['description'].tolist()\nembeddings = st_model.encode(corpus, convert_to_numpy=True, show_progress_bar=True)\nindex = faiss.IndexFlatL2(embeddings.shape[1])\nfaiss.normalize_L2(embeddings)\nindex.add(embeddings)\nfaiss.write_index(index, 'exoplanet.index')\nnp.save('exoplanet.npy', embeddings)\n\n#retrieve relevant information\ndef retrieve(query, k=3):\n    embedding = st_model.encode([query], convert_to_numpy=True, show_progress_bar=False)\n    faiss.normalize_L2(embedding)\n    dst, idx = index.search(embedding, k)\n\n    return df.iloc[idx[0]]['description'].tolist()\n"

RAG strategy 2: define function to retrieve the 3 entries with schematic summary closest to the query by Jaccard similarity

In [11]:
def jaccard_similarity(wordlist1, wordlist2):
    intersection = len(list(set(wordlist1).intersection(set(wordlist2))))
    union = len(list(set(wordlist1).union(set(wordlist2))))
    return float(intersection) / float(union)

def my_distance(a, b):
    # convert to lowercase and remove punctuation
    a = a.lower().translate(str.maketrans(dict.fromkeys(string.punctuation)))
    b = b.lower().translate(str.maketrans(dict.fromkeys(string.punctuation)))
    return jaccard_similarity(a.split(), b.split())

# retrieve relevant information
def retrieve(query, k=3):
    dst = df['summary'].apply(lambda x: my_distance(query, x))
    idx = np.argsort(dst)[-k:]
    
    return df['summary'][idx].to_list()

In [12]:
retrieve("Any planet discovered in 2020?")

['name OGLE-2018-BLG-1700L-b | host_name OGLE-2018-BLG-1700L | discovery_year 2020 | mass 1400.0 | orbital_period unknown | radius unknown | temperature nan',
 'name KMT-2018-BLG-0029L-b | host_name KMT-2018-BLG-0029L | discovery_year 2020 | mass 7.59 | orbital_period unknown | radius unknown | temperature nan',
 'name OGLE-2012-BLG-0838L-b | host_name OGLE-2012-BLG-0838L | discovery_year 2020 | mass 53.1 | orbital_period unknown | radius unknown | temperature nan']

Load flan-t5-large

In [13]:
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

Create example for prompt

In [14]:
example = (f"Context:\n\n{df['summary'].to_list()[2]}\n\n"
            f"{df['summary'].to_list()[1]}\n\n"
            f"{df['summary'].to_list()[0]}\n\n"
            f"Question:\n\nWhat do you know about {df['pl_name'].to_list()[0]}?\n\n"
            f"Answer:\n\n{df['description'].to_list()[0]}\n\n"
           )

In [15]:
print(example)

Context:

name TOI-4513-b | host_name TOI-4513 | discovery_year 2026 | mass unknown | orbital_period 9.76042253684 | radius 2.34587608 | temperature nan

name TOI-4511-b | host_name TOI-4511 | discovery_year 2026 | mass unknown | orbital_period 10.98090968596 | radius 2.75304932 | temperature nan

name TOI-4495-b | host_name TOI-4495 | discovery_year 2026 | mass 7.7 | orbital_period 2.56699 | radius 2.483 | temperature 1735.0

Question:

What do you know about TOI-4495-b?

Answer:

The planet TOI-4495-b orbits around the star TOI-4495. It was discovered in 2026. It has a mass of 7.7 Mass Earth Units. It has an orbital period of 2.56699 days. It has a radius of 2.483 Radius Earth Units. It has a temperature 1735.0 K




Define function that creates a prompt from the question, feeds it to flan-t5 and returns the answer

In [16]:
def reply(question):

    context  = "\n\n".join(retrieve(question))

    prompt = f"Question:\n\n{question}\n\nContext:\n\n{context}\n\nAnswer:\n\n"

    complete_prompt = example + prompt

    #print(f"=== BEGIN PROMPT ===\n{complete_prompt}\n=== END PROMPT ===")

    inputs = tokenizer(complete_prompt, return_tensors="pt")

    ilen = inputs['input_ids'].shape[1]
    if ilen > 512:
        print(f"WARNING: input length is {ilen} > 512.")
    
    output = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.20,
        top_p=0.95,
        no_repeat_ngram_size=3,
        eos_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

Test #1

In [17]:
reply("What is the radius of Kepler-1604-b and when was it discovered?")

'Kepler-1604-b was discovered in 2016 and has a radius of 1.41.'

In [18]:
retrieve("What is the radius of Kepler-1604-b and when was it discovered?")

['name LkCa-15-c | host_name LkCa-15 | discovery_year 2015 | mass unknown | orbital_period unknown | radius unknown | temperature nan',
 'name MWC-758-c | host_name MWC-758 | discovery_year 2023 | mass unknown | orbital_period unknown | radius unknown | temperature 500.0',
 'name Kepler-1604-b | host_name Kepler-1604 | discovery_year 2016 | mass unknown | orbital_period 0.683684256 | radius 1.41 | temperature nan']

In [19]:
df[df['pl_name']=='Kepler-1604-b']['summary'].to_list()

['name Kepler-1604-b | host_name Kepler-1604 | discovery_year 2016 | mass unknown | orbital_period 0.683684256 | radius 1.41 | temperature nan']

Test #2

In [20]:
reply("Any planet discovered in 2010? What is its temperature?")

'The temperature of the planet HAT-P-16-b is 1566.0. It was discovered in 2010. It has a mass of unknown and an orbital period of unknown. Its radius is 14.68376472.'

In [21]:
retrieve("Any planet discovered in 2010? What is its temperature?")

['name SR-12-AB-c | host_name SR-12-AB | discovery_year 2010 | mass 4131.62 | orbital_period unknown | radius unknown | temperature nan',
 'name HIP-78530-b | host_name HIP-78530 | discovery_year 2010 | mass 7300.0 | orbital_period unknown | radius unknown | temperature 2700.0',
 'name HAT-P-16-b | host_name HAT-P-16 | discovery_year 2010 | mass unknown | orbital_period unknown | radius 14.68376472 | temperature 1566.0']

In [22]:
df[df['pl_name']=='HAT-P-16-b']['summary'].to_list()

['name HAT-P-16-b | host_name HAT-P-16 | discovery_year 2010 | mass unknown | orbital_period unknown | radius 14.68376472 | temperature 1566.0']

Test #3

In [23]:
reply("Any planets with a mass between 0.9 and 1.1?")

'Yes'

In [24]:
reply("Can you give me an example of a planet with mass between 0.9 and 1.1?")

'Kepler-59-b, discovered in 2012, has a mass of unknown. It has an orbital period of 11.8681707 and a radius of 1.1.'

The LLM cannot answer correctly these questions because it cannot retrieve enough information

In [25]:
retrieve("Can you give me an example of a planet with mass between 0.9 and 1.1?")

['name K2-156-b | host_name K2-156 | discovery_year 2018 | mass unknown | orbital_period 0.813143 | radius 1.1 | temperature nan',
 'name Kepler-1406-b | host_name Kepler-1406 | discovery_year 2016 | mass unknown | orbital_period 11.62905828 | radius 1.1 | temperature nan',
 'name Kepler-59-b | host_name Kepler-59 | discovery_year 2012 | mass unknown | orbital_period 11.8681707 | radius 1.1 | temperature nan']

Test #4

In [26]:
reply("Can you give me an example of a planet discovered in 2014 with mass between 0.9 and 1.1?")

'Kepler-102-b was discovered in 2014 and has a mass of 1.1. It has an orbital period of 5.286965 and a radius of 0.46. Its host name is KepLER-102.'

In [27]:
retrieve("Can you give me an example of a planet discovered in 2014 with mass between 0.9 and 1.1?")

['name Kepler-392-c | host_name Kepler-392 | discovery_year 2014 | mass unknown | orbital_period 10.423118 | radius 1.1 | temperature nan',
 'name Kepler-374-c | host_name Kepler-374 | discovery_year 2014 | mass unknown | orbital_period 3.282807 | radius 1.1 | temperature nan',
 'name Kepler-102-b | host_name Kepler-102 | discovery_year 2014 | mass 1.1 | orbital_period 5.286965 | radius 0.46 | temperature 857.0']

In [28]:
df[df['pl_name']=='Kepler-102-b']['summary'].to_list()

['name Kepler-102-b | host_name Kepler-102 | discovery_year 2014 | mass 1.1 | orbital_period 5.286965 | radius 0.46 | temperature 857.0']

Test #5

In [29]:
reply("Which planets were discovered last year (in 2025)?")

'The OGLE-2015-BLG-1609L-b, KMT-2023-BLGN-0119L-B, and KMT-BLZ-0119-b were discovered in 2025.'

In [30]:
retrieve("Which planets were discovered last year (in 2025)?")

['name KMT-2023-BLG-1896L-b | host_name KMT-2023-BLG-1896L | discovery_year 2025 | mass 16.35 | orbital_period unknown | radius unknown | temperature nan',
 'name OGLE-2015-BLG-1609L-b | host_name OGLE-2015-BLG-1609L | discovery_year 2025 | mass 76.27881768 | orbital_period unknown | radius unknown | temperature nan',
 'name KMT-2023-BLG-0119L-b | host_name KMT-2023-BLG-0119L | discovery_year 2025 | mass 83.7 | orbital_period unknown | radius unknown | temperature nan']

In [31]:
df[df['pl_name']=='OGLE-2015-BLG-1609L-b']['summary'].to_list()

['name OGLE-2015-BLG-1609L-b | host_name OGLE-2015-BLG-1609L | discovery_year 2025 | mass 76.27881768 | orbital_period unknown | radius unknown | temperature nan']

Fine-tune the model with LoRA (due to hardware constraints, we switch to flan-t5-base and train on a tiny subset of the original dataset)

In [32]:
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [33]:
idx0 = 0
idx1 = 5

Build training dataset for fine-tuning the LLM

In [34]:
# host star
df_train = df[idx0:idx1][['pl_name', 'hostname']].copy()
df_train['pl_name'] = df[idx0:idx1].apply(lambda x: f"What is the host star of planet {x.pl_name}?", axis=1)
df_train['hostname'] = df[idx0:idx1].apply(lambda x: f"The host star of {x.pl_name} is {x.hostname}.", axis=1)
df_train.columns = ['question', 'answer']

# host star
df_test = df[idx0:idx1][['pl_name', 'hostname']].copy()
df_test['pl_name'] = df[idx0:idx1].apply(lambda x: f"Which star does {x.pl_name} orbit?", axis=1)
df_test['hostname'] = df[idx0:idx1].apply(lambda x: f"The host star of {x.pl_name} is {x.hostname}.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# discovery date
df_test = df[idx0:idx1][['pl_name', 'disc_year']].copy()
df_test['pl_name'] = df[idx0:idx1].apply(lambda x: f"When was {x.pl_name} discovered?", axis=1)
df_test['disc_year'] = df[idx0:idx1].apply(lambda x: f"Planet {x.pl_name} was discovered in year {x.disc_year}.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# discovery date
df_test = df[idx0:idx1][['pl_name', 'disc_year']].copy()
df_test['pl_name'] = df[idx0:idx1].apply(lambda x: f"In which year was planet {x.pl_name} discovered?", axis=1)
df_test['disc_year'] = df[idx0:idx1].apply(lambda x: f"Planet {x.pl_name} was discovered in year {x.disc_year}.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# orbital period
df_test = df[idx0:idx1][['pl_name', 'pl_orbper']].copy()
df_test['pl_name'] = df[idx0:idx1].apply(lambda x: f"What is the orbital period of {x.pl_name}?", axis=1)
df_test['pl_orbper'] = df[idx0:idx1].apply(lambda x: f"The orbital period of {x.pl_name} is {x.pl_orbper} days.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# orbital period
df_test = df[idx0:idx1][['pl_name', 'pl_orbper']].copy()
df_test['pl_name'] = df[idx0:idx1].apply(lambda x: f"Period of planet {x.pl_name} in days?", axis=1)
df_test['pl_orbper'] = df[idx0:idx1].apply(lambda x: f"The orbital period of {x.pl_name} is {x.pl_orbper} days.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# radius
df_test = df[idx0:idx1][['pl_name', 'pl_rade']].copy()
df_test['pl_name'] = df[idx0:idx1].apply(lambda x: f"What is the radius of {x.pl_name}?", axis=1)
df_test['pl_rade'] = df[idx0:idx1].apply(lambda x: f"The radius of {x.pl_name} is {x.pl_rade} Earth Radius Units.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# radius
df_test = df[idx0:idx1][['pl_name', 'pl_rade']].copy()
df_test['pl_name'] = df[idx0:idx1].apply(lambda x: f"Radius of planet {x.pl_name} in Earth Radius Units?", axis=1)
df_test['pl_rade'] = df[idx0:idx1].apply(lambda x: f"The radius of {x.pl_name} is {x.pl_rade} Earth Radius Units.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# mass
df_test = df[idx0:idx1][['pl_name', 'pl_bmasse']].copy()
df_test['pl_name'] = df[idx0:idx1].apply(lambda x: f"What is the mass of {x.pl_name}?", axis=1)
df_test['pl_bmasse'] = df[idx0:idx1].apply(lambda x: f"The mass of {x.pl_name} is {x.pl_bmasse} Earth Mass Units.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# mass
df_test = df[idx0:idx1][['pl_name', 'pl_bmasse']].copy()
df_test['pl_name'] = df[idx0:idx1].apply(lambda x: f"Mass of planet {x.pl_name} in Earth Mass Units?", axis=1)
df_test['pl_bmasse'] = df[idx0:idx1].apply(lambda x: f"The mass of {x.pl_name} is {x.pl_bmasse} Earth Mass Units.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# temperature
df_test = df[idx0:idx1][['pl_name', 'pl_eqt']].copy()
df_test['pl_name'] = df[idx0:idx1].apply(lambda x: f"What is the temperature of {x.pl_name}?", axis=1)
df_test['pl_eqt'] = df[idx0:idx1].apply(lambda x: f"The temperature of {x.pl_name} is {x.pl_eqt} K.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# temperature
df_test = df[idx0:idx1][['pl_name', 'pl_eqt']].copy()
df_test['pl_name'] = df[idx0:idx1].apply(lambda x: f"Temperature of planet {x.pl_name} in K?", axis=1)
df_test['pl_eqt'] = df[idx0:idx1].apply(lambda x: f"The temperature of {x.pl_name} is {x.pl_eqt} K.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

df_train['answer'] = df_train['answer'].astype(str)

df_train['question'] = df_train['question'].replace(" nan ", " unknown ", regex=True)
df_train['answer'] = df_train['answer'].replace(" nan ", " unknown ", regex=True)

In [35]:
df_train.shape

(60, 2)

In [36]:
df_train.head(3)

,question,answer
0,What is the host star of planet TOI-4495-b?,The host star of TOI-4495-b is TOI-4495.
1,What is the host star of planet TOI-4511-b?,The host star of TOI-4511-b is TOI-4511.
2,What is the host star of planet TOI-4513-b?,The host star of TOI-4513-b is TOI-4513.


In [37]:
df_train = df_train.sample(df_train.shape[0], random_state=42)

In [38]:
df_train.head(3)

,question,answer
0,What is the host star of planet TOI-4495-b?,The host star of TOI-4495-b is TOI-4495.
5,Which star does TOI-4495-b orbit?,The host star of TOI-4495-b is TOI-4495.
36,Radius of planet TOI-4511-b in Earth Radius Un...,The radius of TOI-4511-b is 2.75304932 Earth R...


Fine-tune the model using LoRA

In [39]:
def print_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}")
    
lora_config = LoraConfig(
    r = 24,
    lora_alpha = 32,
    target_modules = ["q", "v"],
    lora_dropout = 0.05,
    bias = "none",
    task_type = TaskType.SEQ_2_SEQ_LM
)

lora_model = get_peft_model(model, lora_config)

print_trainable_parameters(lora_model)

training_args = Seq2SeqTrainingArguments(
    output_dir="lora-flan-t5-base-exoplanet",
    learning_rate=1.0e-3,
    num_train_epochs=120,
    logging_strategy="epoch",
    save_strategy="epoch",
    push_to_hub=False
)

train_dataset = Dataset.from_pandas(df_train[["question", "answer"]])
train_dataset = train_dataset.rename_columns({"question": "input", "answer": "target"})

max_input_length = 64
max_target_length = 128

def preprocess_batch(batch):
    inputs = tokenizer(batch["input"], truncation=True, padding="longest", max_length=max_input_length)
    targets = tokenizer(batch["target"], truncation=True, padding="longest", max_length=max_target_length)
    return {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "labels": targets["input_ids"],
    }

tokenized_train_dataset = train_dataset.map(preprocess_batch, batched=True, remove_columns=train_dataset.column_names)
tokenized_train_dataset.set_format(type="torch")

data_collator = DataCollatorForSeq2Seq(tokenizer, model=lora_model, label_pad_token_id=-100)

def compute_exact_match(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels == -100, tokenizer.pad_token_id, labels)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    matches = [int(p == l) for p, l in zip(decoded_preds, decoded_labels)]

    return {"exact_match": float(np.mean(matches))}

trainer = Seq2SeqTrainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    data_collator=data_collator,
    compute_metrics=compute_exact_match,
)

trainer.train()

trainable params: 2654208 || all params: 250232064 || trainable%: 1.0606986001602097


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Step,Training Loss
8,10.257504
16,6.025687
24,2.968736
32,2.086518
40,1.747026
48,1.547759
56,1.410589
64,1.295685
72,1.139194
80,0.945564


TrainOutput(global_step=960, training_loss=0.366019988215218, metrics={'train_runtime': 3627.4893, 'train_samples_per_second': 1.985, 'train_steps_per_second': 0.265, 'total_flos': 185137179033600.0, 'train_loss': 0.366019988215218, 'epoch': 120.0})

Define function that creates a prompt from the question, feeds it to the fine-tuned flan-t5 and returns the answer

In [40]:
def lora_reply(question):

    prompt = f"input:\n\n{question}\n\n\n\target:\n\n"

    complete_prompt = prompt

    #print(f"=== BEGIN PROMPT ===\n{complete_prompt}\n=== END PROMPT ===")

    inputs = tokenizer(complete_prompt, return_tensors="pt")

    ilen = inputs['input_ids'].shape[1]
    if ilen > 512:
        print(f"WARNING: input length is {ilen} > 512.")
    
    output = lora_model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
#        temperature=0.20,
#        top_p=0.95,
        no_repeat_ngram_size=3,
        eos_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [41]:
lora_model.eval();

In [42]:
print("model.training =", lora_model.training)

model.training = False


Test #1

In [43]:
lora_reply(f"Can you tell me the radius of {df.iloc[idx0]['pl_name']}?")

'The radius of TOI-4495-b is 2.483 Earth Radius Units.'

In [44]:
df[df['pl_name']==df.iloc[idx0]['pl_name']]['summary'].to_list()

['name TOI-4495-b | host_name TOI-4495 | discovery_year 2026 | mass 7.7 | orbital_period 2.56699 | radius 2.483 | temperature 1735.0']

Test #2

In [45]:
lora_reply(f"Tell me the temperature of {df.iloc[idx0+1]['pl_name']}?")

'The temperature of TOI-4511-b is unknown K.'

In [46]:
df[df['pl_name']==df.iloc[idx0+1]['pl_name']]['summary'].to_list()

['name TOI-4511-b | host_name TOI-4511 | discovery_year 2026 | mass unknown | orbital_period 10.98090968596 | radius 2.75304932 | temperature nan']

Test #3

In [47]:
lora_reply(f"Can you remember the orbital period of {df.iloc[idx0+2]['pl_name']}?")

'The orbital period of TOI-4513-b is 9.76042253684 days.'

In [48]:
df[df['pl_name']==df.iloc[idx0+2]['pl_name']]['summary'].to_list()

['name TOI-4513-b | host_name TOI-4513 | discovery_year 2026 | mass unknown | orbital_period 9.76042253684 | radius 2.34587608 | temperature nan']

Test #4

In [49]:
lora_reply(f"What is the mass of planet {df.iloc[idx0+3]['pl_name']}?")

'The mass of TOI-4529-b is 4.9 Earth Mass Units.'

In [50]:
df[df['pl_name']==df.iloc[idx0+3]['pl_name']]['summary'].to_list()

['name TOI-4529-b | host_name TOI-4529 | discovery_year 2026 | mass 4.9 | orbital_period 5.879577 | radius 1.77 | temperature 511.0']

Test #5

In [51]:
lora_reply(f"What is the planet with the largest mass, {df.iloc[idx0+3]['pl_name']} or {df.iloc[idx0+4]['pl_name']}?")

'The mass of TOI-4529-b is 1.77 Earth Mass Units.'

In [52]:
df[df['pl_name']==df.iloc[idx0+3]['pl_name']]['summary'].to_list()

['name TOI-4529-b | host_name TOI-4529 | discovery_year 2026 | mass 4.9 | orbital_period 5.879577 | radius 1.77 | temperature 511.0']

In [53]:
df[df['pl_name']==df.iloc[idx0+4]['pl_name']]['summary'].to_list()

['name TOI-4552-b | host_name TOI-4552 | discovery_year 2026 | mass 1.83 | orbital_period 0.30110032 | radius 1.11 | temperature 1122.0']

Test #6

In [54]:
lora_reply(f"What is the planet with the largest mass, {df.iloc[idx0]['pl_name']} or {df.iloc[idx0+4]['pl_name']}?")

'The mass of TOI-4495-b is 7.7 Earth Mass Units.'

In [55]:
df[df['pl_name']==df.iloc[idx0]['pl_name']]['summary'].to_list()

['name TOI-4495-b | host_name TOI-4495 | discovery_year 2026 | mass 7.7 | orbital_period 2.56699 | radius 2.483 | temperature 1735.0']

In [56]:
df[df['pl_name']==df.iloc[idx0+4]['pl_name']]['summary'].to_list()

['name TOI-4552-b | host_name TOI-4552 | discovery_year 2026 | mass 1.83 | orbital_period 0.30110032 | radius 1.11 | temperature 1122.0']